# Advanced Experimentation Walkthrough

This notebook reproduces the primary onboarding experiment, inspects realized MDE and power-at-effect, and demonstrates a CUPED-style pre-treatment covariate adjustment.

**Methodological note:** because this is a new-user onboarding experiment, there is no genuine pre-period activation outcome. The variance-reduction example therefore uses a treatment-blind propensity score built only from acquisition channel and device. The unadjusted primary analysis remains confirmatory.

In [ ]:
from src.generate_dataset import generate_users
from src.experiment import two_proportion_test
from src.diagnostics import minimum_detectable_effect, required_sample_size, two_sided_power
from src.cuped import cuped_adjust_activation


In [ ]:
rows = generate_users(12_000, seed=20_260_808)
primary = two_proportion_test(rows, 'activated_7d')
n_control = sum(row['variant'] == 'control' for row in rows)
n_treatment = sum(row['variant'] == 'treatment' for row in rows)
print(f'control: {n_control:,} users @ {primary.control_rate:.2%}')
print(f'treatment: {n_treatment:,} users @ {primary.treatment_rate:.2%}')
print(f'raw lift: {primary.absolute_lift * 100:+.2f} pp; p={primary.p_value:.4f}')


In [ ]:
mde = minimum_detectable_effect(primary.control_rate, len(rows), power=0.80)
observed_power = two_sided_power(primary.control_rate, primary.absolute_lift, n_control, n_treatment)
sample_for_2pp, _ = required_sample_size(primary.control_rate, 0.02, power=0.80)
print(f'80% power MDE: {mde * 100:.2f} pp')
print(f'planning power at observed lift: {observed_power:.1%}')
print(f'users per arm for +2.00 pp at 80% power: {sample_for_2pp:,}')


In [ ]:
cuped = cuped_adjust_activation(rows)
print(f'theta: {cuped.theta:.4f}')
print(f'covariate variance reduction: {cuped.variance_reduction:.2%}')
print(f'raw lift: {cuped.raw_difference * 100:+.2f} pp')
print(f'adjusted lift: {cuped.adjusted_difference * 100:+.2f} pp')
print(f'adjusted p-value: {cuped.p_value:.4f}')


## Decision discipline

- The original unadjusted 7-day activation analysis remains the confirmatory decision statistic.
- MDE and power are planning tools, not post-hoc significance filters.
- The CUPED-style analysis is a precision/sensitivity demonstration, not a replacement for the pre-specified primary analysis.
- In production, freeze the covariate and adjustment method before reading treatment outcomes.